# FLAG 2027 - Notebook 29: voice ATTRIBUTE blocks (Vox-Profile age-sex and accent) on top of system C

Research round 5: within-gender discrimination relies on age, accent and similar traits. The audeering age-gender model was our single largest voice gain (-0.7). Vox-Profile (Feng et al. 2025, arXiv 2505.14648) releases two stronger trait models on WavLM-large (LoRA-fine-tuned on CommonVoice / TIMIT / VoxCeleb and accent corpora; no Bengali-specific training; used FROZEN and declared):

| Block | Model | Output we keep |
|---|---|---|
| `vpas` | tiantiaf/wavlm-large-age-sex | 256-d pooled feature (+ age and sex logits kept as `vpas_ag`) |
| `vpac` | tiantiaf/wavlm-large-narrow-accent (16 English accents incl. South Asia) | 256-d pooled feature (+ 16 accent logits as `vpac_ag`) |

Stage 1 (this notebook): extract on the challenge train + dev files and all MAV rows (~42k clips, ~1.5 h), CV on top of system C (CLIP | R4 + Whisper + NAP-8): + vpas, + vpac, + both. Reference copied from NB28 / NB27. **Adopt** if overall <= -0.4 or gender <= -0.5 with std <= +0.2; stage 2 would then be the VoxCeleb extraction (NB26 recipe) and NB20 v5. About 2 h on a T4, resumable. Paste the **SUMMARY**.

In [ ]:
# Cell 1 [v2 4-Nov] - Drive, GPU (hard check), packages, Vox-Profile code (github.com/tiantiaf0627/vox-profile-release, used only to load the two public models)
# v2: Vox-Profile patches WavLM internals written for transformers 4.46.3 (its requirements.txt); newer versions fail with
# "WavLMEncoderLayerStableLayerNorm.forward() got an unexpected keyword argument 'output_hidden_states'". So we pin 4.46.3.
# FIRST RUN: this cell installs it and RESTARTS the runtime automatically ("session crashed" message is expected). Then run Cell 1 AGAIN and continue.
import importlib.metadata as md, os
if md.version('transformers') != '4.46.3':
    os.system('pip -q install transformers==4.46.3 tokenizers==0.20.3 speechbrain peft loralib')
    print('transformers 4.46.3 installed -> restarting the runtime now. When it reconnects, run Cell 1 again.')
    os.kill(os.getpid(), 9)
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/voxprofile'):
    os.system('git clone -q https://github.com/tiantiaf0627/vox-profile-release /content/voxprofile')
import torch, shutil, sys, transformers
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
sys.path.append('/content/voxprofile/src/model/age_sex'); sys.path.append('/content/voxprofile/src/model/accent')
print('[v2 4-Nov] Device:', torch.cuda.get_device_name(0), '| transformers', transformers.__version__, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - voice file lists in our row order + alignment check with the official ECAPA (identical to NB24 Cell 2)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def wav_token(tokens):
    t = [x for x in tokens if x.lower().endswith('.wav')]; assert len(t) == 1, tokens; return t[0]
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk, mlang = [], [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, vm)); mspk.append(ver + '_' + t[0].split('/')[0]); mlang.append(lf.split('_')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            _ = fr[rng.integers(len(fr))]
            if need:
                zf.extract(vn, D)
            rows.append((ver + '_' + pid, os.path.join(D, vn), lang_))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
rng = np.random.default_rng(1)
for k, saved in {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}.items():
    idx = rng.choice(len(saved), 100, replace=False)
    new = np.stack([ECAPA.encode_batch(torch.from_numpy(librosa.load(FILES[k][i], sr=16000, mono=True)[0]).unsqueeze(0)).squeeze().cpu().numpy() for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('alignment %-6s median cosine %.4f' % (k, np.median(cos))); assert np.median(cos) > 0.98
del ECAPA; torch.cuda.empty_cache()
note('Files: ' + ', '.join('%s %d' % (k, len(v)) for k, v in FILES.items()))

In [ ]:
# Cell 3 - extract the two Vox-Profile blocks (frozen; 256-d pooled feature + logits) on every file; resumable per block
ORDER = SPLIT_NAMES + ['mav', 'extra']
allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
def prep(p, max_sec=15, min_sec=3):
    """16 kHz mono, centre crop to 15 s (the models' training limit), tile up to 3 s (their reliability floor)"""
    w = librosa.load(p, sr=16000, mono=True)[0]; L = 16000 * max_sec
    if len(w) > L:
        w = w[(len(w) - L) // 2:(len(w) - L) // 2 + L]
    if len(w) < 16000 * min_sec:
        w = np.tile(w, int(np.ceil(16000 * min_sec / max(len(w), 1))))[:16000 * min_sec]
    return torch.from_numpy(w).float().unsqueeze(0).to(dev)
def enc_vp(paths, which):
    if which == 'vpas':
        from wavlm_demographics import WavLMWrapper
        m = WavLMWrapper.from_pretrained('tiantiaf/wavlm-large-age-sex').to(dev).eval()
    else:
        from wavlm_accent import WavLMWrapper
        m = WavLMWrapper.from_pretrained('tiantiaf/wavlm-large-narrow-accent').to(dev).eval()
    feats, logits = [], []
    with torch.no_grad():
        for p in tqdm(paths, desc=which):
            out = m(prep(p), return_feature=True)          # (..., pooled feature) - the last element is the 256-d feature
            feats.append(out[-1].reshape(-1).float().cpu().numpy())
            logits.append(torch.cat([o.reshape(-1).float() for o in out[:-1]]).cpu().numpy())
    del m; torch.cuda.empty_cache(); return np.stack(feats), np.stack(logits)
def _l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def sanity(name, E):
    rng = np.random.default_rng(7); idx = rng.choice(len(FILES['train']), 300, replace=False); Z = _l2(E[idx]); lab = y[idx]; C = Z @ Z.T
    same = (lab[:, None] == lab[None, :]) & ~np.eye(len(idx), dtype=bool); diff = lab[:, None] != lab[None, :]
    note('%s sanity: same-id cosine %.3f | different-id %.3f | gap %.3f' % (name, C[same].mean(), C[diff].mean(), C[same].mean() - C[diff].mean()))
for which in ['vpas', 'vpac']:
    p = os.path.join(FEATDIR, which + '.npz')
    if os.path.exists(p):
        print(which, 'already extracted'); continue
    t0 = time.time()
    try:
        F_, G_ = enc_vp(allp[:2], which); print(which, 'smoke test: feature', F_.shape, 'logits', G_.shape)   # loader problems show in seconds, not after an hour
        F_, G_ = enc_vp(allp, which); d = split_back(F_); dg = split_back(G_); sanity(which, d['train'])
        np.savez(p, **d, **{s + '_ag': dg[s] for s in ORDER})
        note('%-5s dim %4d (+%d logits) | %d files in %.1f min' % (which, F_.shape[1], G_.shape[1], len(allp), (time.time() - t0) / 60))
    except Exception as e:
        note('ENCODER %s FAILED: %r' % (which, repr(e)[:400])); torch.cuda.empty_cache()

In [ ]:
# Cell 4 - CV on top of system C: + vpas, + vpac, + both (5 folds x 3 seeds; reference copied from NB28/NB27; harness identical to NB24)
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk'])
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper', 'vpas', 'vpac']:
    pth = os.path.join(FEATDIR, name + '.npz')
    if not os.path.exists(pth):
        print('missing', name); continue
    d = np.load(pth); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; C_VOICE = R4 + ['whisper']
COMBOS = {'E0_C_ref': C_VOICE}
if 'vpas' in FEAT:
    COMBOS['E1_+vpas'] = C_VOICE + ['vpas']
if 'vpac' in FEAT:
    COMBOS['E2_+vpac'] = C_VOICE + ['vpac']
if 'vpas' in FEAT and 'vpac' in FEAT:
    COMBOS['E3_+both'] = C_VOICE + ['vpas', 'vpac']
CSV = os.path.join(RESDIR, 'nb29_cv.csv')
if not os.path.exists(CSV):
    for src, combo in [('nb28_cv.csv', 'H0_C_ref'), ('nb27_cv.csv', 'G0_C_ref')]:
        sp = os.path.join(RESDIR, src)
        if os.path.exists(sp):
            r = pd.read_csv(sp); r = r[r.combo == combo][['combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
            if len(r) == K * len(SEEDS):
                r['combo'] = 'E0_C_ref'; r.to_csv(CSV, index=False); print('reference copied from', src); break
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
F_RAW = build_train(['clip'])
for name, vn_ in COMBOS.items():
    V_full = build_train(vn_); U, _ = nap_basis(V_full[N_CH:], NAP_K); V_raw = nap_apply(V_full, U)
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]; model, best_ep, XF, XV, _ = train_combo(F_RAW, V_raw, tr_idx, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
            print('%-10s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean().round(2); ref = tab.loc['E0_C_ref']
note('VOICE ATTRIBUTE BLOCKS on system C (challenge + all MAV, CLIP | R4 + Whisper + NAP-8; 5-fold x 3 seeds)'); note(tab.to_string())
verdicts = []
for c in tab.index:
    if c == 'E0_C_ref':
        continue
    d_all, d_std, d_gen = tab.loc[c, 'overall'] - ref['overall'], tab.loc[c, 'std'] - ref['std'], tab.loc[c, 'gender'] - ref['gender']
    verdicts.append('%s: overall %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, d_all, d_std, d_gen, 'ADOPT (stage 2 = VoxCeleb extraction + NB20 v5)' if (d_all <= -0.4) or (d_gen <= -0.5 and d_std <= 0.2) else 'reject'))
note('VERDICTS: ' + ' | '.join(verdicts))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)